# 연습 문제 1-11 풀이: 지수 파라미터, 파라미터별 학습률

1-3절 연습 문제 1-11의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 방법으로도 구현할 수 있다.

- 본문 예제 코드는 `code_examples/ch01/` 아래 예제 노트북을 참고한다.

In [1]:
# torch 라이브러리 임포트와 시드 고정
import torch

SEED = 8
torch.manual_seed(SEED)
print(f'시드 고정: {SEED}')

시드 고정: 8


## 사전 준비

본문 1-3절의 데이터 생성, 모델, 손실 함수, 학습 루프([코드 1-28] ~ [코드 1-32])를 가져온다.
에포크마다 로그를 출력하면 실험 조합마다 수백 줄이 나오므로, 마지막 손실만 돌려주는 `train_quiet()`를 따로 구현해 사용한다.

In [2]:
SAMPLE_SIZE = 50    # 생성할 샘플의 수
LR = 0.0001         # 학습률
EPOCHS = 15         # 전체 학습 에포크
INIT = [0.96, -0.56]  # 파라미터 초깃값: 본문 [코드 1-33]과 같은 값

def generate_true_data(sample_size=SAMPLE_SIZE, true_a=4.9, true_b=0.0):
    X = torch.rand((sample_size, 1)) * 10                    # 관측시간
    Y_ideal = true_a * X ** 2 + true_b                       # 이상적인 낙하거리
    noise = torch.randn((sample_size, 1)) * Y_ideal * 0.1    # 관측 오차
    return X, Y_ideal + noise

X, Y_true = generate_true_data()
TRAIN_SIZE = int(SAMPLE_SIZE * 0.8)
X_train, Y_train = X[:TRAIN_SIZE], Y_true[:TRAIN_SIZE]
X_test, Y_test = X[TRAIN_SIZE:], Y_true[TRAIN_SIZE:]
print(f'훈련 데이터 {X_train.shape[0]}개, 평가 데이터 {X_test.shape[0]}개')

훈련 데이터 40개, 평가 데이터 10개


In [3]:
def model(parameters, X):
    return parameters[0] * X ** 2 + parameters[1]

def criterion(Y_pred, Y_true):
    return torch.mean((Y_pred - Y_true) ** 2)

def train_quiet(parameters, X, Y_true, epochs=EPOCHS, learning_rate=LR, criterion=criterion):
    """[코드 1-32]와 같은 학습 루프. 로그 대신 마지막 손실만 반환한다."""
    for _ in range(epochs):
        parameters.grad = None
        train_loss = criterion(model(parameters, X), Y_true)
        train_loss.backward()
        with torch.no_grad():
            parameters -= learning_rate * parameters.grad
    return train_loss.item()

## 연습 문제 1-11 [도전 문제]

> 1-3절의 예제는 딥러닝으로 데이터를 y = ax^2 + b에 맞춰 a와 b의 값을 찾는 회귀 분석 모델이다.
> 그런데 이 데이터가 몇 차의 함수로 표현되는 모델에 적합한지 미리 알 수 없다면 지수까지 파라미터로 두는 모델을
> 사용할 수 있다. 회귀 분석 모델을 y = ax^c + b로 바꾸고 a, b, c 세 파라미터의 값을 찾도록 예제를 수정해 보자.
>
> 힌트: 세 파라미터의 기울기를 출력해 크기를 비교해 보고, 모든 파라미터에 같은 학습률을 쓸 때의 문제점을 생각해 보자.

**모델 함수를 y = ax^c + b로 바꾸고 기울기 비교하기**
- 파라미터는 [a, b, c] 세 값을 담은 텐서이며, 지수 c도 자동 미분 대상
- 초깃값은 [0.96, -0.56, 1.5]: a, b는 본문과 같고, c는 2가 아닌 값에서 출발
- 첫 기울기의 크기를 파라미터별로 비교

In [4]:
def model_exp(parameters, X):
    return parameters[0] * X ** parameters[2] + parameters[1]    # y = a * x^c + b

INIT3 = [0.96, -0.56, 1.5]    # a, b, c의 초깃값
parameters = torch.tensor(INIT3, requires_grad=True)
criterion(model_exp(parameters, X_train), Y_train).backward()
for name, grad in zip('abc', parameters.grad.tolist()):
    print(f'{name}의 기울기: {grad:12.2f}')

a의 기울기:     -5637.64
b의 기울기:      -274.91
c의 기울기:    -11176.76


**결과 분석**
- c의 기울기가 가장 크다: b의 약 40배, a의 약 2배
- 지수는 조금만 바뀌어도 예측값이 몇 배로 달라져서 손실에 미치는 영향이 크다
- 같은 학습률이면 기울기가 클수록 한 번에 많이 움직인다

**모든 파라미터에 같은 학습률(0.0001)로 학습하면?**
- 본문과 같은 학습 루프에 모델 함수만 바꾸고, 처음 4에포크의 손실과 파라미터를 출력
- 15에포크를 마친 뒤의 파라미터도 확인

In [5]:
parameters = torch.tensor(INIT3, requires_grad=True)
print(f'{"에포크":>5} {"손실":>10} {"a":>10} {"b":>10} {"c":>10}')
for epoch in range(1, EPOCHS + 1):
    parameters.grad = None
    train_loss = criterion(model_exp(parameters, X_train), Y_train)
    train_loss.backward()
    if epoch <= 4:
        a, b, c = parameters.tolist()
        print(f'{epoch:8d} {train_loss.item():12.2f} {a:10.4f} {b:10.4f} {c:10.4f}')
    with torch.no_grad():
        parameters -= LR * parameters.grad
print(f'{EPOCHS}에포크 뒤: {parameters.tolist()}')

  에포크         손실          a          b          c
       1     38525.21     0.9600    -0.5600     1.5000
       2      1543.06     1.5238    -0.5325     2.6177
       3     43658.81     0.8730    -0.5326     0.3016
       4     43524.29     0.9261    -0.5033     0.3933
15에포크 뒤: [nan, nan, nan]


**결과 분석: 손실이 오르내리다 nan이 된다**
- 첫 걸음에서 c가 1.5에서 2.6으로 튀었다가 0.3으로 되돌아간다
- 기울기가 큰 c에 a, b와 같은 학습률은 한 걸음이 너무 크다
- 결국 값이 `nan`이 되어 학습이 되지 않는다

**파라미터마다 다른 학습률 적용하기**
- 학습률도 [a, b, c] 세 값을 담은 텐서로 두고, 기울기에 곱해 뺀다
- a, b는 본문과 같은 0.0001, c만 100분의 1인 0.000001
- c가 천천히 움직이므로 에포크를 2,000으로 늘린다

In [17]:
# 파라미터별로 학습률을 다르게 적용하는 학습 함수
def train_exp(init, learning_rates, epochs):
    parameters = torch.tensor(init, requires_grad=True)
    for _ in range(epochs):
        parameters.grad = None
        train_loss = criterion(model_exp(parameters, X_train), Y_train)
        train_loss.backward()
        with torch.no_grad():
            parameters -= learning_rates * parameters.grad    # 파라미터마다 다른 학습률
    return parameters.tolist(), train_loss.item()

(a, b, c), loss = train_exp(INIT3, torch.tensor([0.0001, 0.0001, 0.000001]), 2000)
print(f'a = {a:.4f}, b = {b:.4f}, c = {c:.4f}, 훈련 손실 = {loss:.2f}')

a = 5.7087, b = -0.7796, c = 1.9491, 훈련 손실 = 375.26


**결과 분석: c가 2에 가까워진다**
- c = 1.95: 데이터를 y = 4.9x^2에서 만들었으므로 2에 가까워지는 것이 맞다
- a = 5.71: c가 2보다 조금 작아 x^c가 줄어든 만큼 a가 커져서 보정한다
- 훈련 손실 375는 지수를 2로 고정한 본문 모델(380)과 비슷한 수준

**직접 해 보기: c의 학습률만 10배(0.00001)로 키우면?**
- c가 더 빨리 2에 가까워질까?
- 먼저 a, b, c가 어떻게 될지 예측해 보자

In [18]:
(a, b, c), loss = train_exp(INIT3, torch.tensor([0.0001, 0.0001, 0.00001]), 2000)
print(f'a = {a:.4f}, b = {b:.4f}, c = {c:.4f}, 훈련 손실 = {loss:.2f}')

a = 8.0878, b = 16.0424, c = 1.0366, 훈련 손실 = 6324.05


**확인**
- `nan`은 아니지만 a, b, c가 모두 엉뚱한 값(a = 8.1, b = 16.0, c = 1.0)에 머문다
- 훈련 손실도 6,324로, 앞의 375보다 훨씬 높다
- 학습률이 크면 `nan`으로 끝나지 않고 잘못된 곳에 정착하기도 하므로 손실을 함께 확인해야 한다